# 🎬 SRT Subtitle Generator
### Tạo phụ đề SRT từ audio/video sử dụng Groq Whisper API

Notebook này có thể chạy trên **Google Colab** hoặc **local**.

---

## 1. Cài đặt Dependencies

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q groq gradio

## 2. Cấu hình API Key

Lấy API key tại: https://console.groq.com/keys

In [ ]:
import os

# Cách 1: Nhập trực tiếp (không khuyến khích cho production)
# os.environ["GROQ_API_KEY"] = "your_api_key_here"

# Cách 2: Sử dụng Google Colab Secrets (khuyến khích)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Loaded API key from Colab Secrets")
except:
    print("ℹ️ Running locally - make sure GROQ_API_KEY is set")

# Kiểm tra API key
if os.environ.get("GROQ_API_KEY"):
    print("✅ GROQ_API_KEY is configured")
else:
    print("⚠️ GROQ_API_KEY is not set - please configure it above")

## 3. Core Functions

In [ ]:
import os
from groq import Groq

# Whisper supported languages
SUPPORTED_LANGUAGES = {
    "Auto Detect": "",
    "English": "en",
    "Vietnamese": "vi",
    "Japanese": "ja",
    "Korean": "ko",
    "Chinese": "zh",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Russian": "ru",
    "Arabic": "ar",
    "Hindi": "hi",
    "Thai": "th",
    "Indonesian": "id",
    "Malay": "ms",
    "Turkish": "tr",
    "Polish": "pl",
    "Ukrainian": "uk",
    "Dutch": "nl",
    "Swedish": "sv",
    "Norwegian": "no",
    "Danish": "da",
    "Finnish": "fi",
    "Greek": "el",
    "Hebrew": "he",
    "Czech": "cs",
    "Romanian": "ro",
    "Hungarian": "hu",
    "Filipino": "tl",
}

AVAILABLE_MODELS = [
    "whisper-large-v3-turbo",
    "whisper-large-v3",
]


def format_timestamp(seconds: float) -> str:
    """Convert seconds to SRT timestamp format: HH:MM:SS,mmm"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def transcribe_audio(
    filepath: str,
    model: str = "whisper-large-v3-turbo",
    language: str = "",
    prompt: str = "",
) -> dict:
    """Transcribe audio file using Groq Whisper API."""
    client = Groq()
    
    with open(filepath, "rb") as file:
        params = {
            "file": (os.path.basename(filepath), file.read()),
            "model": model,
            "response_format": "verbose_json",
            "timestamp_granularities": ["segment"],
            "temperature": 0,
        }
        
        if language:
            params["language"] = language
        if prompt:
            params["prompt"] = prompt
        
        transcription = client.audio.transcriptions.create(**params)
    
    return transcription


def segments_to_srt(segments: list) -> str:
    """Convert transcription segments to SRT format."""
    srt_lines = []
    
    for i, segment in enumerate(segments, 1):
        start_time = format_timestamp(segment.get("start", 0))
        end_time = format_timestamp(segment.get("end", 0))
        text = segment.get("text", "").strip()
        
        srt_lines.append(f"{i}")
        srt_lines.append(f"{start_time} --> {end_time}")
        srt_lines.append(text)
        srt_lines.append("")
    
    return "\n".join(srt_lines)


def generate_srt(
    filepath: str,
    model: str = "whisper-large-v3-turbo",
    language: str = "",
    prompt: str = "",
) -> tuple:
    """Generate SRT content from audio file."""
    transcription = transcribe_audio(
        filepath=filepath,
        model=model,
        language=language,
        prompt=prompt,
    )
    
    # Get segments from response
    segments = []
    if hasattr(transcription, "segments"):
        segments = transcription.segments
    elif isinstance(transcription, dict) and "segments" in transcription:
        segments = transcription["segments"]
    
    # Convert segments to list of dicts
    segment_list = []
    for seg in segments:
        if hasattr(seg, "start"):
            segment_list.append({
                "start": seg.start,
                "end": seg.end,
                "text": seg.text,
            })
        else:
            segment_list.append(seg)
    
    srt_content = segments_to_srt(segment_list)
    full_text = transcription.text if hasattr(transcription, "text") else ""
    
    return srt_content, full_text


def save_srt(srt_content: str, output_path: str) -> str:
    """Save SRT content to file."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(srt_content)
    return output_path


print("✅ Functions loaded successfully!")

## 4. Sử dụng trực tiếp (không cần UI)

Upload file audio lên Colab hoặc đặt đường dẫn file local.

In [ ]:
# Upload file trên Colab
try:
    from google.colab import files
    print("📁 Chọn file audio/video để upload...")
    uploaded = files.upload()
    audio_file = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {audio_file}")
except:
    # Local: đặt đường dẫn file
    audio_file = "audio.mp3"  # Thay đổi đường dẫn này
    print(f"ℹ️ Using local file: {audio_file}")

In [ ]:
# Tạo SRT
print(f"🔄 Đang transcribe: {audio_file}...")

srt_content, full_text = generate_srt(
    filepath=audio_file,
    model="whisper-large-v3-turbo",  # hoặc "whisper-large-v3"
    language="",  # để trống = auto detect, hoặc nhập "vi", "en", "ja"...
    prompt="",  # optional context
)

print("\n📄 Nội dung SRT:")
print("-" * 50)
print(srt_content[:2000])  # In 2000 ký tự đầu
if len(srt_content) > 2000:
    print("\n... (còn nữa)")

In [ ]:
# Lưu file SRT
output_file = audio_file.rsplit(".", 1)[0] + ".srt"
save_srt(srt_content, output_file)
print(f"✅ Đã lưu: {output_file}")

# Download file (Colab only)
try:
    from google.colab import files
    files.download(output_file)
    print("⬇️ Đang download...")
except:
    print(f"ℹ️ File đã lưu tại: {output_file}")

---

## 5. Chạy Web UI với Gradio

Chạy cell dưới để mở giao diện web.

In [ ]:
import tempfile
import gradio as gr

def process_audio(audio_file, model, language, prompt):
    """Process audio file and generate SRT."""
    if audio_file is None:
        return "", None, "⚠️ Vui lòng upload file audio/video"
    
    lang_code = SUPPORTED_LANGUAGES.get(language, "")
    
    try:
        srt_content, full_text = generate_srt(
            filepath=audio_file,
            model=model,
            language=lang_code,
            prompt=prompt,
        )
        
        # Save to temp file
        base_name = os.path.splitext(os.path.basename(audio_file))[0]
        temp_dir = tempfile.gettempdir()
        srt_path = os.path.join(temp_dir, f"{base_name}.srt")
        save_srt(srt_content, srt_path)
        
        return srt_content, srt_path, f"✅ Thành công! ({len(srt_content.splitlines())} dòng)"
    
    except Exception as e:
        return "", None, f"❌ Lỗi: {str(e)}"


# Build Gradio UI
with gr.Blocks(
    title="SRT Generator",
    theme=gr.themes.Soft(primary_hue="blue"),
) as app:
    
    gr.Markdown("# 🎬 SRT Subtitle Generator\n### Tạo phụ đề từ audio/video")
    
    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(label="📁 Upload Audio/Video", type="filepath")
            model_dropdown = gr.Dropdown(
                label="🤖 Model",
                choices=AVAILABLE_MODELS,
                value="whisper-large-v3-turbo",
            )
            language_dropdown = gr.Dropdown(
                label="🌐 Ngôn ngữ",
                choices=list(SUPPORTED_LANGUAGES.keys()),
                value="Auto Detect",
            )
            prompt_input = gr.Textbox(label="📝 Prompt (tùy chọn)", lines=2)
            generate_btn = gr.Button("🚀 Generate SRT", variant="primary")
        
        with gr.Column():
            status_output = gr.Textbox(label="📊 Trạng thái", interactive=False)
            srt_output = gr.Textbox(label="📄 Nội dung SRT", lines=15, show_copy_button=True)
            download_output = gr.File(label="⬇️ Download SRT")
    
    generate_btn.click(
        fn=process_audio,
        inputs=[audio_input, model_dropdown, language_dropdown, prompt_input],
        outputs=[srt_output, download_output, status_output],
    )

# Launch
app.launch(share=True)  # share=True để tạo public URL trên Colab

---

## 📚 Tham Khảo

- **Groq Console**: https://console.groq.com
- **Groq Docs**: https://console.groq.com/docs/speech-to-text
- **Supported Languages**: Whisper hỗ trợ 100+ ngôn ngữ
- **File Size Limit**: 25MB (free) / 100MB (dev tier)